In [0]:
# Databricks Notebook: L3_Oro_Hypertuning.py
# ==============================================================================
# CAPA GOLD - ANÁLISIS DE HIPERTUNING (Ejecutar después de L3_Oro.py)
# Analiza diferentes valores de K y guarda resultados para visualización
# ==============================================================================

import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Inicializar Spark Session
spark = SparkSession.builder.appName("OroHypertuning").getOrCreate()

# Configuración de esquemas
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

print("="*80)
print("🔧 CAPA GOLD: Análisis de Hipertuning (K=3 a K=7)")
print("="*80)

# ==============================================================================
# PASO 1: CARGAR TABLA RFM Y PREPROCESAR
# ==============================================================================

print("\n📂 Paso 1: Cargando datos y preprocesando...")

# Cargar RFM desde Silver
rfm_spark = spark.table(f"{SILVER_SCHEMA}.rfm_features")
rfm_logistica = rfm_spark.toPandas()

print(f"   ✅ Cargados {len(rfm_logistica):,} clientes")

# Features para análisis (sin Antigüedad)
features_tuning = [
    'Recencia', 'Frecuencia', 'Monetario',
    'Amplitud_Categorias', 'Total_Articulos',
    'Avg_Peso_g', 'Avg_Volumen_cm3', 'Avg_Installments'
]

# Pre-procesamiento estándar
data_tuning_raw = rfm_logistica[features_tuning].fillna(0)
data_tuning_log = np.log1p(data_tuning_raw)

scaler_tuning = StandardScaler()
data_tuning_scaled = scaler_tuning.fit_transform(data_tuning_log)

print(f"   Datos listos. Dimensiones: {data_tuning_scaled.shape}")

# ==============================================================================
# PASO 2: EXPERIMENTACIÓN CON DIFERENTES VALORES DE K
# ==============================================================================

print("\n🧪 Paso 2: Ejecutando experimentos (K=3 a K=7)...")

resultados_tecnicos = []
modelos = {}  # Guardamos las etiquetas de cada modelo
k_range = range(3, 8)

for k in k_range:
    print(f"   > Entrenando K-Means con K={k}...", end=" ")
    
    # Entrenar modelo
    kmeans = KMeans(
        n_clusters=k, 
        init='k-means++', 
        n_init=10, 
        max_iter=300, 
        random_state=42
    )
    kmeans.fit(data_tuning_scaled)
    
    # Calcular métricas técnicas
    inertia = kmeans.inertia_
    sil_score = silhouette_score(data_tuning_scaled, kmeans.labels_)
    
    # Calcular distribución de clusters
    unique, counts = np.unique(kmeans.labels_, return_counts=True)
    cluster_distribution = dict(zip(unique, counts))
    
    # Guardar resultados
    resultados_tecnicos.append({
        'K': k,
        'Inercia': inertia,
        'Silueta': sil_score,
        'Cluster_0_Size': cluster_distribution.get(0, 0),
        'Cluster_1_Size': cluster_distribution.get(1, 0),
        'Cluster_2_Size': cluster_distribution.get(2, 0) if k >= 3 else None,
        'Cluster_3_Size': cluster_distribution.get(3, 0) if k >= 4 else None,
        'Cluster_4_Size': cluster_distribution.get(4, 0) if k >= 5 else None,
        'Cluster_5_Size': cluster_distribution.get(5, 0) if k >= 6 else None,
        'Cluster_6_Size': cluster_distribution.get(6, 0) if k >= 7 else None
    })
    
    # Guardar etiquetas para análisis de negocio
    modelos[k] = kmeans.labels_
    
    print(f"✅ (Silueta: {sil_score:.4f}, Inercia: {inertia:.2f})")

print(f"\n   ✅ Completados {len(resultados_tecnicos)} experimentos")

# ==============================================================================
# PASO 3: ANÁLISIS DE NEGOCIO POR CADA K
# ==============================================================================

print("\n📊 Paso 3: Analizando perfiles de negocio para cada K...")

resumen_comparativo = []

for k in k_range:
    # Asignar etiquetas temporales
    rfm_logistica[f'Segmento_K{k}'] = modelos[k]
    
    # Calcular promedios para este K
    perfil = rfm_logistica.groupby(f'Segmento_K{k}')[['Monetario', 'Frecuencia', 'Recencia']].mean()
    
    # Identificar segmento "Premium" (mayor frecuencia)
    id_premium = perfil['Frecuencia'].idxmax()
    datos_premium = perfil.loc[id_premium]
    
    # Calcular tamaño del segmento premium
    tamaño_premium = (rfm_logistica[f'Segmento_K{k}'] == id_premium).sum()
    pct_premium = (tamaño_premium / len(rfm_logistica)) * 100
    
    # Identificar segmento "Ballena" (mayor monetario)
    id_ballena = perfil['Monetario'].idxmax()
    datos_ballena = perfil.loc[id_ballena]
    tamaño_ballena = (rfm_logistica[f'Segmento_K{k}'] == id_ballena).sum()
    pct_ballena = (tamaño_ballena / len(rfm_logistica)) * 100
    
    # Identificar segmento "Nuevo" (menor recencia)
    id_nuevo = perfil['Recencia'].idxmin()
    datos_nuevo = perfil.loc[id_nuevo]
    tamaño_nuevo = (rfm_logistica[f'Segmento_K{k}'] == id_nuevo).sum()
    pct_nuevo = (tamaño_nuevo / len(rfm_logistica)) * 100
    
    resumen_comparativo.append({
        'K': k,
        
        # Segmento Premium
        'Premium_ID': id_premium,
        'Premium_Size': tamaño_premium,
        'Premium_Pct': pct_premium,
        'Premium_Frecuencia': datos_premium['Frecuencia'],
        'Premium_Monetario': datos_premium['Monetario'],
        'Premium_Recencia': datos_premium['Recencia'],
        
        # Segmento Ballena
        'Ballena_ID': id_ballena,
        'Ballena_Size': tamaño_ballena,
        'Ballena_Pct': pct_ballena,
        'Ballena_Monetario': datos_ballena['Monetario'],
        'Ballena_Frecuencia': datos_ballena['Frecuencia'],
        
        # Segmento Nuevo
        'Nuevo_ID': id_nuevo,
        'Nuevo_Size': tamaño_nuevo,
        'Nuevo_Pct': pct_nuevo,
        'Nuevo_Recencia': datos_nuevo['Recencia']
    })
    
    print(f"   K={k}: Premium={pct_premium:.1f}%, Ballena={pct_ballena:.1f}%, Nuevo={pct_nuevo:.1f}%")

# ==============================================================================
# PASO 4: IDENTIFICAR MEJOR K
# ==============================================================================

print("\n🏆 Paso 4: Identificando mejor K...")

df_tecnico = pd.DataFrame(resultados_tecnicos)

# Mejor K por Silueta
best_k_silhouette = df_tecnico.loc[df_tecnico['Silueta'].idxmax()]
print(f"   Mejor K por Silueta: K={int(best_k_silhouette['K'])} (Score: {best_k_silhouette['Silueta']:.4f})")

# Mejor K por Inercia (método del codo - heurística)
# Calculamos la "curvatura" (segunda derivada)
inercias = df_tecnico['Inercia'].values
deltas = np.diff(inercias)
curvatures = np.diff(deltas)
elbow_idx = np.argmax(np.abs(curvatures)) + 2  # +2 por dos diffs
elbow_k = df_tecnico.iloc[elbow_idx]['K']

print(f"   Codo detectado en: K={int(elbow_k)}")

# Recomendación final
print(f"\n   📌 Recomendación: K=5 (criterio de negocio basado en análisis)")

# ==============================================================================
# PASO 5: GUARDAR RESULTADOS EN TABLAS GOLD
# ==============================================================================

print(f"\n💾 Paso 5: Guardando resultados en {GOLD_SCHEMA}...")

# Tabla 1: Métricas Técnicas por K
metricas_k_spark = spark.createDataFrame(df_tecnico)
metricas_k_spark.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{GOLD_SCHEMA}.hypertuning_metrics")

print(f"   ✅ Tabla 1: {GOLD_SCHEMA}.hypertuning_metrics ({len(df_tecnico)} registros)")

# Tabla 2: Análisis de Negocio por K
df_negocio = pd.DataFrame(resumen_comparativo)
negocio_k_spark = spark.createDataFrame(df_negocio)
negocio_k_spark.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{GOLD_SCHEMA}.hypertuning_business_analysis")

print(f"   ✅ Tabla 2: {GOLD_SCHEMA}.hypertuning_business_analysis ({len(df_negocio)} registros)")

# Tabla 3: Perfil Detallado de Cada K (para comparación)
perfiles_detallados = []

for k in k_range:
    perfil = rfm_logistica.groupby(f'Segmento_K{k}')[features_tuning].mean().reset_index()
    perfil['K'] = k
    perfil.rename(columns={f'Segmento_K{k}': 'Cluster_ID'}, inplace=True)
    
    # Agregar tamaño de cluster
    cluster_sizes = rfm_logistica[f'Segmento_K{k}'].value_counts().reset_index()
    cluster_sizes.columns = ['Cluster_ID', 'Cluster_Size']
    
    perfil = perfil.merge(cluster_sizes, on='Cluster_ID', how='left')
    
    perfiles_detallados.append(perfil)

df_perfiles_detallados = pd.concat(perfiles_detallados, ignore_index=True)

perfiles_spark = spark.createDataFrame(df_perfiles_detallados)
perfiles_spark.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{GOLD_SCHEMA}.hypertuning_cluster_profiles")

print(f"   ✅ Tabla 3: {GOLD_SCHEMA}.hypertuning_cluster_profiles ({len(df_perfiles_detallados)} registros)")

# Tabla 4: Resumen Ejecutivo del Análisis
resumen_ejecutivo = pd.DataFrame([{
    'Fecha_Analisis': pd.Timestamp.now(),
    'Total_Clientes': len(rfm_logistica),
    'K_Minimo': int(k_range[0]),
    'K_Maximo': int(k_range[-1]),
    'Mejor_K_Silueta': int(best_k_silhouette['K']),
    'Mejor_Score_Silueta': float(best_k_silhouette['Silueta']),
    'K_Codo': int(elbow_k),
    'K_Recomendado': 5,
    'Razon_Recomendacion': 'Criterio de negocio: mejor separación de Premium, Ballena y Voluminoso'
}])

resumen_spark = spark.createDataFrame(resumen_ejecutivo)
resumen_spark.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{GOLD_SCHEMA}.hypertuning_summary")

print(f"   ✅ Tabla 4: {GOLD_SCHEMA}.hypertuning_summary (1 registro)")

# ==============================================================================
# PASO 6: MOSTRAR RESULTADOS
# ==============================================================================

print("\n" + "="*80)
print("📊 RESUMEN DEL ANÁLISIS DE HIPERTUNING")
print("="*80)

print("\n📈 Métricas Técnicas por K:")
print(df_tecnico[['K', 'Inercia', 'Silueta']].to_string(index=False))

print("\n💼 Análisis de Negocio (Segmento Premium):")
print(df_negocio[['K', 'Premium_Pct', 'Premium_Frecuencia', 'Premium_Monetario']].to_string(index=False))

print("\n🏆 Recomendaciones:")
print(f"   - Mejor K por Silueta: {int(best_k_silhouette['K'])}")
print(f"   - Codo detectado en: {int(elbow_k)}")
print(f"   - K Recomendado: 5 (mejor balance técnico-negocio)")

print("\n" + "="*80)
print("✅ ANÁLISIS DE HIPERTUNING COMPLETADO")
print("="*80)
print(f"Tablas creadas en {GOLD_SCHEMA}:")
print(f"  1. hypertuning_metrics (métricas técnicas)")
print(f"  2. hypertuning_business_analysis (análisis de negocio)")
print(f"  3. hypertuning_cluster_profiles (perfiles detallados)")
print(f"  4. hypertuning_summary (resumen ejecutivo)")
print("="*80)

# Verificar tablas creadas
print("\n📋 Tablas disponibles:")
spark.sql(f"SHOW TABLES IN {GOLD_SCHEMA}").show()

print("\n*** PROCESO DE HIPERTUNING FINALIZADO EXITOSAMENTE ***")